In [1]:
import random

# --- CONFIGURATION & CONSTANTS ---
POP_SIZE = 20
GENERATIONS = 50
MUTATION_RATE = 0.2
ELITE_SIZE = 2

# Target constraints: Let's optimize frequencies for 3 time-blocks (e.g., Morning, Afternoon, Evening)
# Genes represent headway (minutes between buses). Ideal values depend on demand.
DEMAND = [100, 30, 80]  # Relative passenger demand for Morning, Afternoon, Evening

def fitness_function(timetable):
    """
    Evaluates a timetable.
    Lower score is better (we want to minimize total cost).
    """
    passenger_wait_cost = 0
    operator_cost = 0

    for headway, demand in zip(timetable, DEMAND):
        # Shorter headways = happier passengers (less wait time)
        passenger_wait_cost += (headway / 2) * demand

        # Shorter headways = more buses running = higher cost for operator
        operator_cost += (60 / headway) * 50  # 50 is an arbitrary cost per bus trip

    # Total cost to minimize
    return passenger_wait_cost + operator_cost

# --- GENETIC ALGORITHM STEPS ---

def initialize_population():
    # Generate random intervals between 5 and 60 minutes for the 3 time blocks
    return [[random.randint(5, 60) for _ in range(3)] for _ in range(POP_SIZE)]

def selection(population, fitnesses):
    # Tournament selection: pick 3 random individuals, keep the best one
    selected = random.sample(list(zip(population, fitnesses)), 3)
    selected.sort(key=lambda x: x[1])  # Sort by fitness (ascending, lower is better)
    return selected[0][0]

def crossover(parent1, parent2):
    # Single-point crossover
    point = random.randint(1, 2)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2

def mutate(timetable):
    # Randomly tweak a time block's headway
    if random.random() < MUTATION_RATE:
        idx = random.randint(0, 2)
        timetable[idx] = max(5, min(60, timetable[idx] + random.choice([-5, 5])))
    return timetable

# --- MAIN ENGINE ---

population = initialize_population()

for generation in range(GENERATIONS):
    # 1. Evaluate fitness
    fitnesses = [fitness_function(ind) for ind in population]

    # Sort population by fitness to preserve elites
    pop_fit = sorted(zip(population, fitnesses), key=lambda x: x[1])
    new_population = [ind for ind, fit in pop_fit[:ELITE_SIZE]] # Carry over elites

    # 2. Breed next generation
    while len(new_population) < POP_SIZE:
        p1 = selection(population, fitnesses)
        p2 = selection(population, fitnesses)

        c1, c2 = crossover(p1, p2)

        new_population.append(mutate(c1))
        if len(new_population) < POP_SIZE:
            new_population.append(mutate(c2))

    population = new_population

# --- RESULTS ---
final_fitnesses = [fitness_function(ind) for ind in population]
best_timetable = population[final_fitnesses.index(min(final_fitnesses))]

print(f"Optimized Headways (Morning, Afternoon, Evening): {best_timetable} minutes")
print(f"Minimized Cost Score: {min(final_fitnesses):.2f}")


Optimized Headways (Morning, Afternoon, Evening): [8, 13, 7] minutes
Minimized Cost Score: 1909.34
